In [54]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [55]:
df = pd.read_csv("data/processed/aqi_clean.csv", parse_dates=['Date'])

In [56]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8676 entries, 0 to 8675
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   City    8676 non-null   str           
 1   Date    8676 non-null   datetime64[us]
 2   PM2.5   8676 non-null   float64       
 3   NO2     8676 non-null   float64       
 4   CO      8676 non-null   float64       
 5   SO2     8676 non-null   float64       
 6   O3      8676 non-null   float64       
 7   AQI     8676 non-null   float64       
dtypes: datetime64[us](1), float64(6), str(1)
memory usage: 542.4 KB


In [57]:
df["next_date"] = df["Date"].shift(-1)
print((df["next_date"] - df["Date"]).value_counts())

1 days        8586
2 days          45
4 days          11
3 days          11
5 days           5
19 days          2
13 days          2
18 days          2
-1926 days       1
12 days          1
-2008 days       1
79 days          1
24 days          1
-1919 days       1
21 days          1
-1455 days       1
204 days         1
11 days          1
8 days           1
Name: count, dtype: int64


In [58]:
df['gap'] = df.groupby('City')['Date'].diff().dt.days
print(df.groupby('City')['gap'].value_counts())

City           gap  
Bengaluru      1.0      1825
               3.0         3
               19.0        2
               2.0         2
               13.0        2
               4.0         1
               5.0         1
               18.0        1
Chennai        1.0      1868
               2.0         5
               4.0         3
               3.0         2
               12.0        1
               18.0        1
Delhi          1.0      1903
               24.0        1
               79.0        1
               2.0         1
Hyderabad      1.0      1867
               2.0         7
               5.0         2
               21.0        1
               4.0         1
               3.0         1
Visakhapatnam  1.0      1123
               2.0        30
               4.0         6
               3.0         5
               5.0         2
               204.0       1
               11.0        1
               8.0         1
Name: count, dtype: int64


In [59]:
parts = []
for city, g in df.groupby("City"):
    g = g.set_index("Date").asfreq('D')
    g['City'] = city
    parts.append(g)
daily = pd.concat(parts).reset_index()

In [60]:
df = (df.drop('gap', axis=1) if 'gap' in df.columns else df)

In [61]:
df.columns

Index(['City', 'Date', 'PM2.5', 'NO2', 'CO', 'SO2', 'O3', 'AQI', 'next_date'],
      dtype='str')

In [62]:
num_cols = ['PM2.5', 'NO2', 'CO', 'SO2', 'O3', 'AQI']

In [63]:
daily[num_cols] = daily.groupby('City')[num_cols].transform(
    lambda s: s.interpolate(limit=2, limit_area='inside'))

In [64]:
daily["gap"] = daily.groupby("City")["Date"].diff().dt.days
print(daily.groupby("City")["gap"].value_counts())
print()
print(daily.isnull().sum())
daily = daily.drop(["gap", "next_date"] , axis=1)
print(daily.columns)

City           gap
Bengaluru      1.0    1929
Chennai        1.0    1926
Delhi          1.0    2008
Hyderabad      1.0    1919
Visakhapatnam  1.0    1455
Name: count, dtype: int64

Date           0
City           0
PM2.5        441
NO2          441
CO           441
SO2          441
O3           441
AQI          441
next_date    567
gap            5
dtype: int64
Index(['Date', 'City', 'PM2.5', 'NO2', 'CO', 'SO2', 'O3', 'AQI'], dtype='str')


In [65]:
d = daily[daily.City == 'Delhi']
print(d['AQI'].isnull().sum())

97


In [67]:
daily['AQI_next'] = daily.groupby('City')['AQI'].shift(-1)

In [69]:
for i in ['1', '2', '3', '7']:
    key = "AQI_lag" + i
    daily[key] = daily.groupby("City")["AQI"].shift(int(i))

In [70]:
daily.columns

Index(['Date', 'City', 'PM2.5', 'NO2', 'CO', 'SO2', 'O3', 'AQI', 'AQI_next',
       'AQI_lag1', 'AQI_lag2', 'AQI_lag3', 'AQI_lag7'],
      dtype='str')

In [71]:
daily["AQI_roll7"] = daily.groupby("City")["AQI"].transform(
    lambda s: s.rolling(7).mean()
)


In [78]:
daily['month'] = daily["Date"].dt.month
daily['dayofweek'] = daily['Date'].dt.day_of_week

In [79]:
print(daily[daily.City == 'Delhi'][['Date', 'AQI', 'AQI_lag1', 'AQI_next']].head(10))

           Date    AQI  AQI_lag1  AQI_next
3857 2015-01-01  472.0       NaN     454.0
3858 2015-01-02  454.0     472.0     143.0
3859 2015-01-03  143.0     454.0     319.0
3860 2015-01-04  319.0     143.0     325.0
3861 2015-01-05  325.0     319.0     318.0
3862 2015-01-06  318.0     325.0     353.0
3863 2015-01-07  353.0     318.0     383.0
3864 2015-01-08  383.0     353.0     375.0
3865 2015-01-09  375.0     383.0     376.0
3866 2015-01-10  376.0     375.0     379.0


In [80]:
daily = daily.drop(columns=['gap', 'next_date'], errors="ignore")
print(daily.isnull().sum())
daily = daily.dropna()

Date           0
City           0
PM2.5        441
NO2          441
CO           441
SO2          441
O3           441
AQI          441
AQI_next     446
AQI_lag1     446
AQI_lag2     451
AQI_lag3     456
AQI_lag7     476
AQI_roll7    637
month          0
dayofweek      0
dtype: int64


In [81]:
daily = pd.get_dummies(daily, columns=['City'], prefix='city', dtype=int)


In [82]:
print(daily.shape)
print(daily.isnull().sum().sum())   # must be 0
print(daily.columns.tolist())

(8547, 20)
0
['Date', 'PM2.5', 'NO2', 'CO', 'SO2', 'O3', 'AQI', 'AQI_next', 'AQI_lag1', 'AQI_lag2', 'AQI_lag3', 'AQI_lag7', 'AQI_roll7', 'month', 'dayofweek', 'city_Bengaluru', 'city_Chennai', 'city_Delhi', 'city_Hyderabad', 'city_Visakhapatnam']


In [83]:
daily.head()

,Date,PM2.5,NO2,CO,SO2,O3,AQI,AQI_next,AQI_lag1,AQI_lag2,AQI_lag3,AQI_lag7,AQI_roll7,month,dayofweek,city_Bengaluru,city_Chennai,city_Delhi,city_Hyderabad,city_Visakhapatnam
7,2015-03-28,40.00,17.99,8.07,2.50,44.65,152.0,143.0,123.0,132.0,232.0,91.0,147.428571,3,5,1,0,0,0,0
8,2015-03-29,32.16,12.90,8.53,2.08,56.31,143.0,80.0,152.0,123.0,132.0,120.0,150.714286,3,6,1,0,0,0,0
9,2015-03-30,30.78,16.23,1.46,2.63,49.52,80.0,90.0,143.0,152.0,123.0,154.0,140.142857,3,0,1,0,0,0,0
10,2015-03-31,38.99,10.97,1.79,2.79,47.67,90.0,151.0,80.0,143.0,152.0,119.0,136.000000,3,1,1,0,0,0,0
11,2015-04-01,36.42,8.83,9.60,3.17,48.25,151.0,123.0,90.0,80.0,143.0,232.0,124.428571,4,2,1,0,0,0,0


In [84]:
daily.to_csv("finalized_data.csv", index=False)